# Комбинация: Pipeline + Voting — надёжный конвейер

Обработка юридических документов. Каждый этап конвейера продублирован тремя агентами, работающими параллельно через `Send()`. Результаты агрегируются голосованием большинства, и только консенсусный результат переходит на следующий этап.

Два этапа конвейера, каждый продублирован 3 агентами:
1. **Extraction**: 3 экстрактора → голосование → консенсус
2. **Analysis**: 3 аналитика → голосование → итог

Стоимость — 3x на каждый этап, но для юридических документов цена ошибки несопоставимо выше. Два уровня голосования дают надёжность, недостижимую одиночным агентом.

```mermaid
graph LR
    START((START)) -->|Send x3| E1(["🔹 Экстрактор 1<br/><i>извлекает данные</i>"])
    START -->|Send x3| E2(["🔹 Экстрактор 2<br/><i>извлекает данные</i>"])
    START -->|Send x3| E3(["🔹 Экстрактор 3<br/><i>извлекает данные</i>"])
    E1 --> VE(["📊 Голосование: экстракция<br/><i>подсчёт голосов</i>"])
    E2 --> VE
    E3 --> VE
    VE -->|Send x3| A1(["🔹 Аналитик 1<br/><i>анализирует</i>"])
    VE -->|Send x3| A2(["🔹 Аналитик 2<br/><i>анализирует</i>"])
    VE -->|Send x3| A3(["🔹 Аналитик 3<br/><i>анализирует</i>"])
    A1 --> VA(["📊 Голосование: анализ<br/><i>подсчёт голосов</i>"])
    A2 --> VA
    A3 --> VA
    VA --> END((END))

    classDef coordinator fill:#4A90D9,stroke:#2C5F8A,color:#fff,stroke-width:2px
    classDef worker fill:#2ECC71,stroke:#1A8B4C,color:#fff,stroke-width:2px
    classDef aggregator fill:#F59E0B,stroke:#D97706,color:#fff,stroke-width:2px
    classDef terminal fill:#95A5A6,stroke:#707B7C,color:#fff

    class START,END terminal
    class E1,E2,E3,A1,A2,A3 worker
    class VE,VA aggregator
```

In [1]:
import sys
sys.path.insert(0, "..")

import operator
from typing import TypedDict, Annotated
from collections import Counter

from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from langchain_core.messages import HumanMessage, SystemMessage
from llm_config import get_llm, check_api_key

In [2]:
if not check_api_key():
    raise ValueError("API key is not set")
else:
    print("API key is set")

API key is set


## Два класса состояний

`PipelineState` — состояние конвейера. `WorkerState` — то, что получает каждый параллельный экземпляр агента через `Send()`. Поля `extraction_votes` и `analysis_votes` используют reducer `operator.add` — это точки сбора результатов от параллельных агентов. LangGraph автоматически добавляет каждый новый результат к списку, а не перезаписывает его.

In [3]:
llm = get_llm()

# --- Состояние для параллельного агента ---
class WorkerState(TypedDict):
    text: str
    worker_id: int
    stage: str

# --- Основное состояние ---
class PipelineState(TypedDict):
    document: str
    extraction_votes: Annotated[list[str], operator.add]
    extraction_result: str
    analysis_votes: Annotated[list[str], operator.add]
    analysis_result: str

## Этап 1: Параллельная экстракция и fan-out

Функция `spawn_extractors` — условное ребро, возвращающее список из трёх `Send()`. Каждый `Send` создаёт независимый экземпляр узла `extractor` со своим `WorkerState`. Три экстрактора работают параллельно с разными промптами, и каждый возвращает результат через reducer `extraction_votes`.

После завершения всех трёх — мета-агент `vote_extraction` синтезирует консенсус: выделяет пункты, в которых согласны минимум 2 из 3, и отбрасывает спорные.

In [4]:
def spawn_extractors(state: PipelineState) -> list[Send]:
    print(f"  [Этап 1] Запускаю 3 экстрактора параллельно")
    return [
        Send("extractor", {"text": state["document"], "worker_id": i, "stage": "extraction"})
        for i in range(1, 4)
    ]

def extractor_node(state: WorkerState) -> dict:
    prompts = [
        "Извлеки ключевые обязательства сторон из документа. Кратко: 3-4 пункта.",
        "Найди основные условия и обязательства в документе. Перечисли 3-4 пункта.",
        "Определи главные пункты договорённостей. Выдели 3-4 ключевых.",
    ]
    response = llm.invoke([
        SystemMessage(content=prompts[state["worker_id"] - 1]),
        HumanMessage(content=state["text"])
    ])
    print(f"  [Экстрактор {state['worker_id']}] Готово")
    return {"extraction_votes": [response.content]}

def vote_extraction(state: PipelineState) -> dict:
    # Мета-агент синтезирует консенсус из 3 извлечений
    all_extractions = "\n\n---\n\n".join(
        f"Экстрактор {i+1}:\n{v}" for i, v in enumerate(state["extraction_votes"])
    )
    response = llm.invoke([
        SystemMessage(content=(
            "Ты — арбитр. Перед тобой 3 независимые извлечения из документа. "
            "Выдели пункты, в которых согласны минимум 2 из 3. "
            "Отбрось спорные. Верни консенсусный список."
        )),
        HumanMessage(content=all_extractions)
    ])
    print(f"  [Голосование: extraction] Консенсус из {len(state['extraction_votes'])} извлечений")
    return {"extraction_result": response.content}

## Этап 2: Параллельный анализ с голосованием

Второй этап конвейера устроен идентично: `spawn_analysts` раздаёт консенсусный результат экстракции трём независимым аналитикам через `Send()`. Каждый аналитик оценивает риски со своей перспективы (юридические риски, слабые места, потенциальные проблемы). Арбитр `vote_analysis` оставляет только те риски, которые отмечены минимум двумя аналитиками, и формирует итоговый список с приоритетами.

In [5]:
def spawn_analysts(state: PipelineState) -> list[Send]:
    print(f"  [Этап 2] Запускаю 3 аналитика параллельно")
    return [
        Send("analyst", {"text": state["extraction_result"], "worker_id": i, "stage": "analysis"})
        for i in range(1, 4)
    ]

def analyst_node(state: WorkerState) -> dict:
    prompts = [
        "Оцени юридические риски по пунктам. Кратко: 2-3 риска.",
        "Проанализируй слабые места соглашения. 2-3 пункта.",
        "Найди потенциальные проблемы в условиях. 2-3 замечания.",
    ]
    response = llm.invoke([
        SystemMessage(content=prompts[state["worker_id"] - 1]),
        HumanMessage(content=state["text"])
    ])
    print(f"  [Аналитик {state['worker_id']}] Готово")
    return {"analysis_votes": [response.content]}

def vote_analysis(state: PipelineState) -> dict:
    all_analyses = "\n\n---\n\n".join(
        f"Аналитик {i+1}:\n{v}" for i, v in enumerate(state["analysis_votes"])
    )
    response = llm.invoke([
        SystemMessage(content=(
            "Ты — арбитр. Перед тобой 3 независимых анализа рисков. "
            "Выдели риски, отмеченные минимум 2 аналитиками. "
            "Сформируй итоговый список рисков с приоритетами."
        )),
        HumanMessage(content=all_analyses)
    ])
    print(f"  [Голосование: analysis] Консенсус из {len(state['analysis_votes'])} анализов")
    return {"analysis_result": response.content}

## Сборка графа

Условные рёбра из `START` и из `vote_extraction` используют функции fan-out, которые возвращают списки `Send()`. Каждый `Send` запускает параллельный экземпляр узла. Безусловные рёбра `extractor → vote_extraction` и `analyst → vote_analysis` гарантируют, что голосование начнётся только после завершения всех параллельных экземпляров.

In [6]:
graph = StateGraph(PipelineState)

graph.add_node("extractor", extractor_node)
graph.add_node("vote_extraction", vote_extraction)
graph.add_node("analyst", analyst_node)
graph.add_node("vote_analysis", vote_analysis)

# Этап 1: параллельная экстракция → голосование
graph.add_conditional_edges(START, spawn_extractors)
graph.add_edge("extractor", "vote_extraction")

# Этап 2: параллельный анализ → голосование
graph.add_conditional_edges("vote_extraction", spawn_analysts)
graph.add_edge("analyst", "vote_analysis")

graph.add_edge("vote_analysis", END)

app = graph.compile()

## Запуск

Входной документ — договор аренды с конкретными условиями. Конвейер последовательно пропустит его через два этапа: экстракция ключевых пунктов (3 агента + голосование), затем анализ рисков (3 агента + голосование). Итого 8 вызовов LLM: 3 экстрактора + 1 арбитр + 3 аналитика + 1 арбитр.

In [7]:
result = app.invoke({
    "document": (
        "Договор аренды: Арендодатель предоставляет помещение 120 кв.м. сроком на 2 года. "
        "Арендная плата — 500 000 руб/мес с индексацией 5% ежегодно. "
        "Арендатор обязуется не проводить перепланировку без согласования. "
        "Досрочное расторжение — за 3 месяца с штрафом 2 месячных платежа. "
        "Капитальный ремонт — за счёт арендодателя, текущий — за счёт арендатора."
    ),
    "extraction_votes": [], "extraction_result": "",
    "analysis_votes": [], "analysis_result": "",
})

print(f"\n{'=' * 60}")
print("РЕЗУЛЬТАТ: Консенсус экстракции")
print("=" * 60)
print(result["extraction_result"])

print(f"\n{'=' * 60}")
print("РЕЗУЛЬТАТ: Консенсус анализа рисков")
print("=" * 60)
print(result["analysis_result"])

  [Этап 1] Запускаю 3 экстрактора параллельно


  [Экстрактор 2] Готово
  [Экстрактор 3] Готово
  [Экстрактор 1] Готово


  [Голосование: extraction] Консенсус из 3 извлечений
  [Этап 2] Запускаю 3 аналитика параллельно


  [Аналитик 3] Готово


  [Аналитик 2] Готово


  [Аналитик 1] Готово


  [Голосование: analysis] Консенсус из 3 анализов

РЕЗУЛЬТАТ: Консенсус экстракции
Консенсусный список пунктов, которые подтверждают минимум 2 из 3 экстракторов:

- Арендодатель предоставляет/передаёт Арендатору помещение площадью **120 кв. м** сроком на **2 года**.
- Арендная плата составляет **500 000 руб./мес.** с **ежегодной индексацией на 5%**.
- Арендатор **не вправе проводить перепланировку без согласования** с Арендодателем.
- При **досрочном расторжении** требуется **уведомление за 3 месяца** и предусмотрен **штраф в размере 2 месячных платежей**.
- **Капитальный ремонт** выполняет/оплачивает **Арендодатель**, а **текущий ремонт** — **Арендатор**.

РЕЗУЛЬТАТ: Консенсус анализа рисков
Ниже — риски, которые отметили **минимум 2 аналитика**, с приоритетами.

## Итоговый список рисков

### 1) Риск по досрочному расторжению и штрафу — **высокий приоритет**
**Отмечали:** Аналитик 1, 2, 3  
**Суть:** неясно, кто и в каких случаях может расторгнуть договор досрочно, и как соотносятся 

## Итоги

**Pipeline + Voting** комбинирует два базовых паттерна:
- **Pipeline** задаёт последовательность этапов обработки (экстракция -> анализ)
- **Voting** через `Send()` дублирует каждый этап тремя независимыми агентами и агрегирует результаты голосованием большинства

Ключевые механизмы LangGraph:
- `Send()` — fan-out: создание параллельных экземпляров одного узла с разными входами
- `operator.add` reducer — автоматический сбор результатов от всех экземпляров в единый список
- Условные рёбра, возвращающие `list[Send]` — запуск параллельной обработки между этапами конвейера

Когда применять: задачи, где цена ошибки высока (юридические документы, медицинские заключения, финансовый аудит), и тройная стоимость LLM-вызовов оправдана надёжностью результата.